# Music Review Insight Finder: Comparing Training Approaches

This notebook compares two approaches to improve classification after the initial LoRA experiment:

- **Experiment A:** Uses cross-entropy loss across the three class choices, represented by A, B and C.
- **Experiment B:** Uses the same loss with class weights, giving more importance to less common training categories.

Both experiments started from the same pretrained model with fresh adapters. The initialization seed, LoRA settings, learning rate and three-epoch training schedule were kept the same. Each used **140 training passages and 33 validation passages**. Class weights were calculated from the training data only.

The initial model’s validation macro-F1 was **0.3054**. We used the same validation set to compare the new approaches. The original test set was excluded because it had already been evaluated.

The dataset uses AI-assisted labels with human review. These comparisons supported model development, but a fresh, independently human-labeled evaluation set would be needed to confirm performance on new data.

The experiments ran in Google Colab using a T4 GPU. The saved outputs below document the results.

## 1. Reuse the known working T4 environment

The first run used Python 3.12.13, PyTorch 2.11.0+cu128, and a Tesla T4. Select **Runtime → Change runtime type → T4 GPU**, using the free service; select **2026.07** for the known Python/PyTorch runtime if needed. If it is unavailable, stop rather than purchasing compute automatically.

In [1]:
import os, sys
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_DISABLED"] = "true"
assert (3,10) <= sys.version_info[:2] <= (3,12), "Choose runtime version 2026.07 / Python 3.12."
import torch
assert torch.cuda.is_available(), "Select a free T4 GPU first."
assert "T4" in torch.cuda.get_device_name(0), "This bounded draft uses the known working T4 path."
assert torch.__version__.split("+")[0] == "2.11.0", "Use runtime version 2026.07 for the known working PyTorch build."
assert torch.cuda.get_device_properties(0).total_memory >= 7*1024**3
print(sys.version.split()[0], torch.__version__, torch.cuda.get_device_name(0))

3.12.13 2.11.0+cu128 Tesla T4


In [2]:
import importlib.metadata, subprocess
PINS = {"transformers":"4.57.1", "peft":"0.17.1", "accelerate":"1.10.1", "safetensors":"0.6.2",
        "numpy":"1.26.4", "pandas":"2.2.3", "matplotlib":"3.9.4", "scikit-learn":"1.6.1"}
def installed_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
needs_install = any(installed_version(name) != version for name,version in PINS.items())
SETUP_NEEDS_RESTART = False
if needs_install:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet"] + [name+"=="+v for name,v in PINS.items()])
    SETUP_NEEDS_RESTART = True
    print("Packages installed. Runtime > Restart session NOW, then resume at section 2. Do not Run all.")
else:
    print("Pinned dependencies already match. Continue at section 2; no installation needed.")

Packages installed. Runtime > Restart session NOW, then resume at section 2. Do not Run all.


## 2. Deliberately upload the train/validation-only bundle

Select **music_insight_TRAIN_VALIDATION_ONLY_AB_bundle.zip** to upload the training and validation data to Google Colab. Use this input bundle rather than the original bundle or a results ZIP. If packages were installed, restart the session before continuing. The next cell checks that the files, passage groups, label counts and labeling records match the expected inputs before loading the model.

In [1]:
assert not globals().get("SETUP_NEEDS_RESTART",False), "Restart the session after installation; resume at section 2."
import sys, importlib.metadata
PINS = {"transformers":"4.57.1", "peft":"0.17.1", "accelerate":"1.10.1", "safetensors":"0.6.2",
        "numpy":"1.26.4", "pandas":"2.2.3", "matplotlib":"3.9.4", "scikit-learn":"1.6.1"}
for name,version in PINS.items(): assert importlib.metadata.version(name) == version, "Run section 1 setup first."
from google.colab import files
uploaded = files.upload()
assert len(uploaded)==1, "Select the one train/validation-only bundle."
archive_name = next(iter(uploaded))

Saving music_insight_TRAIN_VALIDATION_ONLY_AB_bundle.zip to music_insight_TRAIN_VALIDATION_ONLY_AB_bundle.zip


In [2]:
from pathlib import Path
import hashlib, io, json, tempfile, zipfile
EXPECTED_BUNDLE_SHA256 = '19309e7694de6b51027b6bbf03e1f8a8a50c66c1f5f533ea56db71f7290ac2b8'
EXPECTED_MEMBERS = {'requirements_colab_validation_only.txt', 'data/manifest.json', 'bundle_sha256.json', 'validation_experiments.py', 'validation_contract.py', 'data/train.csv', 'data/validation.csv'}
EXPECTED_HELPERS = {'validation_contract.py': 'e663fc148bee844f085cc5e644cf5a1f01ab3e6c66d208f19b2a96afa0ee76cb', 'validation_experiments.py': '1cc9590bd7145323653623aa5421eaf365511070e2311a4596290b6cb2cc0988'}
assert hashlib.sha256(uploaded[archive_name]).hexdigest() == EXPECTED_BUNDLE_SHA256, "Wrong or changed archive."
with zipfile.ZipFile(io.BytesIO(uploaded[archive_name])) as z:
    assert len(z.namelist()) == len(EXPECTED_MEMBERS) and set(z.namelist()) == EXPECTED_MEMBERS
    assert sum(info.file_size for info in z.infolist()) < 5_000_000
    blobs = {name:z.read(name) for name in EXPECTED_MEMBERS}
hashes = json.loads(blobs["bundle_sha256.json"])
assert set(hashes) == EXPECTED_MEMBERS-{"bundle_sha256.json"}
for name,digest in hashes.items(): assert hashlib.sha256(blobs[name]).hexdigest() == digest, name
for name,digest in EXPECTED_HELPERS.items(): assert hashlib.sha256(blobs[name]).hexdigest() == digest, name
WORK_DIR = Path(tempfile.mkdtemp(prefix="music_validation_only_",dir="/content"))
for name,blob in blobs.items():
    path=WORK_DIR/name; path.parent.mkdir(parents=True,exist_ok=True); path.write_bytes(blob)
sys.path.insert(0,str(WORK_DIR))
import validation_contract as contract
parts, manifest = contract.load_data(WORK_DIR)
assert set(parts)=={"train","validation"}
print("Verified only training/validation:",{name:len(rows) for name,rows in parts.items()})
print("Class counts:",manifest["class_counts"])
print("AI-assisted labels; original test already viewed; no confirmatory final comparison here.")

Verified only training/validation: {'train': 140, 'validation': 33}
Class counts: {'train': {'general_opinion': 40, 'specific_musical_observation': 79, 'background_other': 21}, 'validation': {'specific_musical_observation': 18, 'general_opinion': 7, 'background_other': 8}}
AI-assisted labels; original test already viewed; no confirmatory final comparison here.


## 3. Consistent Training and Model Checks

Both experiments used the same prompt and represented the three categories as **A, B and C**. The correct answer was kept separate from the input. Training compared the model’s scores for these three choices at the position where it predicts the answer.

Before training, checks confirmed that padding did not affect scoring and that the model could not use a later answer to make its prediction. Training and validation used the same scoring method. Invalid scores were rejected, and output probabilities were not treated as verified confidence levels.

Each experiment started with the same pretrained model and a new LoRA adapter. Checks confirmed that the starting values matched and that only the adapter learned during training. The original model’s weights stayed unchanged. We used the same random seed to make the runs more consistent, though results can vary slightly based on the GPU.

Gradients were accumulated across groups of up to **eight passages** before each update. In the weighted experiment, the loss was normalized across the full group. Training logs recorded completed and skipped updates, gradient values and learning rates to help evaluate the process.

In [3]:
import os, torch, importlib.metadata
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(0)
assert torch.__version__.split("+")[0]=="2.11.0"
from transformers import AutoTokenizer
import validation_experiments as experiments
tokenizer=AutoTokenizer.from_pretrained(experiments.MODEL_ID,revision=experiments.MODEL_REVISION,trust_remote_code=False)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="right"
print("A/B/C token IDs:",dict(zip(contract.CODES,experiments.mapping(tokenizer))))
print("Training-only class weights A/B/C:",contract.class_weights(parts["train"]))
VERSIONS={name:importlib.metadata.version(name) for name in ["torch",*PINS]}
OUTPUT=WORK_DIR/"validation_only_results"; OUTPUT.mkdir(exist_ok=False)
contract.save_json(OUTPUT/"protocol.json",{"experiments":list(experiments.EXPERIMENTS),
    "original_validation_macro_f1":contract.ORIGINAL_VALIDATION_MACRO_F1,
    "bundle_sha256":EXPECTED_BUNDLE_SHA256,"seed":manifest["seed"],
    "model_revision":experiments.MODEL_REVISION,"versions":VERSIONS,
    "selection":"validation macro-F1, then lower unweighted validation CE, then earlier epoch",
    "independent_human_evaluation":False,"original_test_already_viewed":True,
    "scope":"Exactly two three-epoch development runs; no held-out final evaluation"})
print("Original validation comparator:",contract.ORIGINAL_VALIDATION_MACRO_F1)
print("Output folder:",OUTPUT)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

A/B/C token IDs: {'A': 32, 'B': 33, 'C': 34}
Training-only class weights A/B/C: [0.5907172995780591, 1.1666666666666667, 2.2222222222222223]
Original validation comparator: 0.30536659108087677
Output folder: /content/music_validation_only_1n3xkmod/validation_only_results


## 4. Experiment A: Training Without Class Weights

Experiment A trained a new LoRA adapter for **three epochs**, or three passes through the training data. The original model’s weights stayed unchanged. The model learned to choose between **A, B and C**, with each training passage given the same weight.

After each epoch, validation scores were saved, along with prediction counts and a confusion matrix to observe which categories the model performed poorly with. Training logs recorded completed and skipped updates. The best saved version was selected using **validation macro-F1**, which gives each category equal importance. Results were saved without overwriting earlier runs.

In [4]:
summary_A=experiments.run_experiment("A_unweighted_ABC",tokenizer,parts["train"],parts["validation"],manifest,OUTPUT,VERSIONS)
print("A complete. Selected validation macro-F1:",summary_A["selected"]["validation"]["macro_f1"])
print("Successful/skipped updates:",summary_A["successful_optimizer_steps"],summary_A["skipped_optimizer_steps"])

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

A_unweighted_ABC epoch 1 validation macro-F1 0.305367 predictions {'specific_musical_observation': 31, 'general_opinion': 2, 'background_other': 0} updates/skips 16 2
A_unweighted_ABC epoch 2 validation macro-F1 0.235294 predictions {'specific_musical_observation': 33, 'general_opinion': 0, 'background_other': 0} updates/skips 18 0
A_unweighted_ABC epoch 3 validation macro-F1 0.226667 predictions {'specific_musical_observation': 32, 'general_opinion': 1, 'background_other': 0} updates/skips 18 0
A complete. Selected validation macro-F1: 0.30536659108087677
Successful/skipped updates: 52 2


## 5. Experiment B: Training With Class Weights

Experiment B used class weights to give less common categories more influence during training. Errors on these categories received a larger penalty, with the goal of improving performance across all three categories. The weights were calculated using only the training data: **A = 0.591, B = 1.167 and C = 2.222**.

A fresh model and LoRA adapter were used with the same starting values as Experiment A. Training lasted **three epochs**, and all other training settings stayed the same. Labels and data splits were unchanged, and no passages were duplicated. This allowed the comparison to focus on the effect of class weights.

In [5]:
assert summary_A["status"]=="completed_validation_only", "Finish A first."
summary_B=experiments.run_experiment("B_train_weighted_ABC",tokenizer,parts["train"],parts["validation"],manifest,OUTPUT,VERSIONS)
assert summary_A["base_sha256_before"]==summary_B["base_sha256_before"], "Experiments did not start from identical base weights."
assert summary_A["adapter_initial_sha256"]==summary_B["adapter_initial_sha256"], "Adapters did not have identical initialization."
print("B complete. Selected validation macro-F1:",summary_B["selected"]["validation"]["macro_f1"])

B_train_weighted_ABC epoch 1 validation macro-F1 0.297619 predictions {'specific_musical_observation': 3, 'general_opinion': 14, 'background_other': 16} updates/skips 16 2
B_train_weighted_ABC epoch 2 validation macro-F1 0.464881 predictions {'specific_musical_observation': 25, 'general_opinion': 4, 'background_other': 4} updates/skips 18 0
B_train_weighted_ABC epoch 3 validation macro-F1 0.385302 predictions {'specific_musical_observation': 14, 'general_opinion': 14, 'background_other': 5} updates/skips 18 0
B complete. Selected validation macro-F1: 0.464881371858116


## 6. Comparing Validation Results

The table compares Experiments A and B with the original model’s saved validation score. Experiment A matched the original **macro-F1 of 0.305**, while Experiment B reached **0.465**. Accuracy increased from **54.5% in A to 57.6% in B**. These results suggest that class weights helped the model perform better across the three categories on this validation set.

All comparisons used the same **33 validation passages**. Because these passages were used to select the best model, the results are preliminary. A fresh, independently human-labeled evaluation set would help determine whether the improvement also applies to new passages. The original test set was excluded from these experiments.

In [6]:
import pandas as pd
comparison=[{"experiment":"original_run_06","validation_macro_f1":contract.ORIGINAL_VALIDATION_MACRO_F1,"delta_vs_original":0.0}]
for summary in [summary_A,summary_B]:
    metrics=summary["selected"]["validation"]
    comparison.append({"experiment":summary["experiment"],"selected_epoch":summary["selected"]["epoch"],
        "validation_macro_f1":metrics["macro_f1"],"delta_vs_original":summary["delta_vs_original_validation"],
        "validation_accuracy":metrics["accuracy"],"prediction_counts":metrics["prediction_counts"],
        "per_class":metrics["per_class"],"successful_updates":summary["successful_optimizer_steps"],
        "skipped_updates":summary["skipped_optimizer_steps"],"frozen_base_verified":summary["frozen_base_verified"]})
contract.save_json(OUTPUT/"validation_comparison.json",comparison)
display(pd.DataFrame(comparison).drop(columns=["per_class"]))
best_new=max([summary_A,summary_B],key=lambda s:(s["selected"]["validation"]["macro_f1"],-s["selected"]["validation_unweighted_ce"],-s["selected"]["epoch"]))
delta=best_new["delta_vs_original_validation"]
print(("Higher" if delta>0 else "Equal" if delta==0 else "Lower"),"validation macro-F1 versus original:",round(delta,6))
print("Exploratory validation comparison only. Fresh independent holdout needed for a confirmatory final comparison.")
contract.save_json(OUTPUT/"completion.json",{"status":"two_validation_experiments_completed", "runs":[summary_A,summary_B],
    "original_validation_macro_f1":contract.ORIGINAL_VALIDATION_MACRO_F1,"independent_human_evaluation":False,
    "original_test_already_viewed":True,"confirmatory_final_comparison_performed":False})

,experiment,validation_macro_f1,delta_vs_original,selected_epoch,validation_accuracy,prediction_counts,successful_updates,skipped_updates,frozen_base_verified
0,original_run_06,0.305367,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
1,A_unweighted_ABC,0.305367,0.000000,1.0,0.545455,"{'specific_musical_observation': 31, 'general_...",52.0,2.0,True
2,B_train_weighted_ABC,0.464881,0.159515,2.0,0.575758,"{'specific_musical_observation': 25, 'general_...",52.0,2.0,True


Higher validation macro-F1 versus original: 0.159515
Exploratory validation comparison only. Fresh independent holdout needed for a confirmatory final comparison.


In [7]:
# Creates a ZIP in this runtime only; no publication, sharing, or automatic upload.
import shutil
result_archive=shutil.make_archive(str(WORK_DIR/"music_validation_only_AB_results"),"zip",root_dir=OUTPUT)
print("Download this ZIP using Colab's Files pane:",result_archive)
print("Also download the executed notebook. Keep the first run and its results unchanged.")

Download this ZIP using Colab's Files pane: /content/music_validation_only_1n3xkmod/music_validation_only_AB_results.zip
Also download the executed notebook. Keep the first run and its results unchanged.
